# 04 — Baseline: independent FastConformer PC v2

This measures the untouched NVIDIA FastConformer PC model on the locked PC v2 held-out test reciter. The PC model outputs Arabic without diacritics, so canonical scores quantify the full surface gap while quranic_light scores isolate lexical ASR quality. Results are stored separately from archived experiments.

In [ ]:
from pathlib import Path
import os

# Each notebook may open in a fresh Colab runtime, so mount Drive before any
# path check rather than relying on a previous notebook's session.
from google.colab import drive
DRIVE_ROOT = Path("/content/drive/MyDrive")
if not DRIVE_ROOT.exists():
    drive.mount("/content/drive")

# Place the *contents* of this repository in this Google Drive folder, or edit
# this one variable to match the folder you chose.
PROJECT_DIR = DRIVE_ROOT / "quran-fastconformer-colab"
assert PROJECT_DIR.exists(), f"Project directory not found: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())


In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import site
import subprocess
import sys

# Validate real imports, not merely package metadata. Recent Colab images can
# leave NumPy 2.x binary extensions behind after NeMo pins NumPy 1.26.4.
def _numpy_compatible_error():
    try:
        import numpy as np
        if np.__version__ != "1.26.4":
            return f"NumPy {np.__version__} is installed; this project requires 1.26.4"
        import pandas as pd
        from datasets import load_dataset  # noqa: F401
        return None
    except Exception as error:
        return f"binary/import compatibility check failed: {type(error).__name__}: {error}"


def _remove_stale_binary_packages():
    patterns = ("numpy", "numpy-*.dist-info", "pandas", "pandas-*.dist-info")
    for package_dir in site.getsitepackages():
        root = Path(package_dir)
        for pattern in patterns:
            for target in root.glob(pattern):
                if target.is_dir():
                    shutil.rmtree(target, ignore_errors=True)
                else:
                    target.unlink(missing_ok=True)


compatibility_error = _numpy_compatible_error()
required_modules = ("datasets", "jiwer", "soundfile", "yaml", "nemo", "matplotlib")
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]

if compatibility_error:
    print("Repairing incompatible NumPy/pandas binaries:", compatibility_error)
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "numpy", "pandas"])
    _remove_stale_binary_packages()
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "--force-reinstall",
        "numpy==1.26.4",
        "pandas==2.2.3",
    ])
    os.environ["QURAN_COLAB_RESTART_REQUIRED"] = "1"
    print("Clean NumPy repair completed. Use Runtime → Restart session before running any other cell.")
elif missing_modules:
    print("Installing missing project dependencies:", missing_modules)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir", "-r", "requirements.txt"])
    os.environ["QURAN_COLAB_RESTART_REQUIRED"] = "1"
    print("Dependencies updated. Use Runtime → Restart session before launching a NeMo stage.")
else:
    print("Core project dependencies and NumPy binary compatibility are available.")


In [ ]:
!python -m src.baseline --config configs/fastconformer_quran.yaml --manifest artifacts/experiments/fastconformer_pc_v2/manifests/experiment_manifest.json


In [ ]:
import json
from pathlib import Path
metrics = json.loads(Path("artifacts/experiments/fastconformer_pc_v2/results/baseline/metrics.json").read_text(encoding="utf-8"))
print("Held-out reciter(s):", metrics["held_out_test_reciters"])
print(f"Canonical WER / CER: {metrics['strict']['wer_percent']:.2f}% / {metrics['strict']['cer_percent']:.2f}%")
print(f"Lexical (quranic_light) WER / CER: {metrics['diagnostic']['wer_percent']:.2f}% / {metrics['diagnostic']['cer_percent']:.2f}%")
